# Store Cleaned Data in PostgreSQL


### Before Running This Notebook

PostgreSQL must be installed and the `admin` user must be set up with the right privileges.  

The database and user must already exist. Run these **once** in your terminal:

```bash
psql postgres
```

```sql
CREATE DATABASE bank_reviews;
CREATE USER admin WITH PASSWORD '654123';
GRANT ALL PRIVILEGES ON DATABASE bank_reviews TO admin;
GRANT CREATE ON SCHEMA public TO admin;
```

> Postgres 18 is used

## Connection Creation with our database

In [10]:
import psycopg2
import pandas as pd

# Shared connection parameters — change these to match your setup
DB_CONFIG = {
    "host":     "localhost",
    "database": "bank_reviews",
    "user":     "admin_user",
    "password": "654123"
    }


## Schema Creation

In [11]:
conn = None
cur = None

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    print("Connection created!")
except Exception as ex:
    print("Connection Error!")
    print(ex)

# --- bank table ---
# Create bank table
# SERIAL auto-increments the ID on every insert — no need to supply it manually
# UNIQUE ensures no two bank share the same name
try:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS banks (
            bank_id   SERIAL PRIMARY KEY,
            bank_name VARCHAR(255) UNIQUE,
            app_name   VARCHAR(255),
            bank_code   VARCHAR(25) UNIQUE
        );
    """)
    print("Banks Table created successfully!")
except Exception as ex:
    print("Banks Table not created!")
    print(ex)

# --- reviews table (must come after bank because of the foreign key) ---
# Create reviews table
# bank_id is a FOREIGN KEY — it must match an existing bank.bank_id
# This link is what allows us to JOIN the two tables later
try:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS reviews (
            review_id     SERIAL PRIMARY KEY,
            bank_id INT REFERENCES banks(bank_id),
            review_text   TEXT,
            rating        INT,
            review_date   DATE,
            sentiment_label VARCHAR(50),
            sentiment_score NUMERIC(10, 9),
            identified_theme    VARCHAR(50),
            source        VARCHAR(50)
        );
    """)
    print("Reviews Table created successfully!")
except Exception as ex:
    print("Reviews Table not created!")
    print(ex)

conn.commit()
cur.close()
conn.close()

print("Tables created, connection closed successfully.")

Connection created!
Banks Table created successfully!
Reviews Table created successfully!
Tables created, connection closed successfully.


## Inserting data into Banks Database

In [12]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    print("Connection created!")
except Exception as ex:
    print("Connection Error!")
    print(ex)

# Insert banks data
try: 
    cur.execute(
        """
        INSERT INTO banks (bank_name, app_name, bank_code)
        VALUES ('Commercial Bank of Ethiopia', 'CBE Mobile Banking', 'CBE'),
                ('Bank of Abyssinia', 'BoA Mobile', 'Bank of Abyssinia'),
                ('Dashen Bank', 'Dashen Bank Super App', 'Dashen Bank')
                ON CONFLICT DO NOTHING;
        """
    )
    print("Data inserted into Banks Table successfully!")
except Exception as ex:
    print("Data insertion into Banks Table failed!")
    print(ex)

conn.commit()

Connection created!
Data inserted into Banks Table successfully!


## Review processed data before inserting to DB

In [13]:
df_reviews = pd.read_csv("../data/processed/fintech_sentiment_analysis_results.csv")

print(f"Reviews:     {len(df_reviews)} rows")
print()
print(df_reviews)

Reviews:     1390 rows

                                 review_id         bank  \
0     68dd84a3-07ab-41d9-aff8-ad7916ff71a8  Dashen Bank   
1     78284116-8313-4ca8-a28c-61ce4fa44ee6  Dashen Bank   
2     b4a7ea95-727b-4501-8d84-77db88187c84  Dashen Bank   
3     59a20f9e-bb87-4d5f-bee7-ce19f19baddf  Dashen Bank   
4     fc184115-ab13-482b-bb08-798589a4d482  Dashen Bank   
...                                    ...          ...   
1385  dabed99c-87c9-49bc-bf32-c0d7fb205ee0          CBE   
1386  6e90d496-9fe8-469a-bd7c-431615321a0b          CBE   
1387  cd638255-2fb3-4772-98f9-b90d8732dc05          CBE   
1388  5338162f-86ba-47c8-ba55-8d8d57b169f8          CBE   
1389  446d14c5-2a58-49df-8035-dae42208656b          CBE   

                                                 review  rating        date  \
0                            best app so far. thank you       5  2026-05-15   
1     Very Annoying App i tried to open virtual bank...       1  2026-05-14   
2                             

## Inserting processed data into Reviews DB

In [14]:
cur.execute("SELECT * FROM banks;")

print(cur.fetchall())

[(4, 'Commercial Bank of Ethiopia', 'CBE Mobile Banking', 'CBE'), (5, 'Bank of Abyssinia', 'BoA Mobile', 'Bank of Abyssinia'), (6, 'Dashen Bank', 'Dashen Bank Super App', 'Dashen Bank')]


#### Mapping Banks table id with Bank code

In [15]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    print("Connection created!")
except Exception as ex:
    print("Connection Error!")
    print(ex)

    
cur.execute("SELECT bank_id, bank_code FROM banks;")

bank_mapping = {
    code: bank_id
    for bank_id, code in cur.fetchall()
}

print(bank_mapping)

Connection created!
{'CBE': 4, 'Bank of Abyssinia': 5, 'Dashen Bank': 6}


## data-loading into PostgreSQL 

In [16]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    print("Connection created!")
except Exception as ex:
    print("Connection Error!")
    print(ex)

# --- Insert reviews ---
try: 

    for _, row in df_reviews.iterrows():
        bank_id = bank_mapping.get(row["bank"])
        cur.execute(
            """
            INSERT INTO reviews ( bank_id, review_text, rating, review_date, sentiment_label, sentiment_score, identified_theme, source )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT DO NOTHING;
            """,
            (
                bank_id,
                row["review"],
                int(row["rating"]),
                row["date"],
                row["sentiment"],
                row["transformer_sentiment_score"],
                row["theme"],
                row["source"]
            )
        )

    print(f"Inserted {len(df_reviews)} review rows.")
except Exception as ex:
    print("Data Insertion failed!")
    print(ex)
# --- Save and close ---
conn.commit()   # permanently write all inserts to disk
cur.close()
conn.close()

print("Done — connection closed.")

Connection created!
Inserted 1390 review rows.
Done — connection closed.


## View loaded data

In [17]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    print("Connection created!")
except Exception as ex:
    print("Connection Error!")
    print(ex)

try:
    cur.execute("SELECT * FROM reviews limit 5;")
    print(cur.fetchall())
except Exception as ex:
    print("unable to display data")
    print(ex)

Connection created!
[(1, 6, 'best app so far. thank you', 5, datetime.date(2026, 5, 15), 'positive', Decimal('0.999848008'), 'Other', 'Google Play'), (2, 6, 'Very Annoying App i tried to open virtual bank account with fayda but in the end it says something went wrong i tried so many times it says something went wrong why???fix it quickly for now i give you 1/5👎👎👎', 1, datetime.date(2026, 5, 14), 'negative', Decimal('-0.999443710'), 'Account', 'Google Play'), (3, 6, 'good', 5, datetime.date(2026, 5, 14), 'positive', Decimal('0.999816120'), 'Other', 'Google Play'), (4, 6, 'good', 5, datetime.date(2026, 5, 14), 'positive', Decimal('0.999816120'), 'Other', 'Google Play'), (5, 6, 'good app but it was doesnt work other bank transfer and require so many updates why', 5, datetime.date(2026, 5, 14), 'positive', Decimal('-0.996920347'), 'UX', 'Google Play')]
